In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers datasets torch sacrebleu underthesea

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 116.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

#Fine tune trên mô hình pretrained

In [ ]:
import torch
from transformers import MarianMTModel, MarianTokenizer
from transformers import Trainer, TrainingArguments
from datasets import Dataset
import sacrebleu
import os

# 1. Đọc dữ liệu từ file
def load_parallel_data(en_file, vi_file):
    with open(en_file, 'r', encoding='utf-8') as f_en, open(vi_file, 'r', encoding='utf-8') as f_vi:
        en_lines = [line.strip() for line in f_en]
        vi_lines = [line.strip() for line in f_vi]
    # Đảm bảo số dòng khớp nhau
    assert len(en_lines) == len(vi_lines), "Số dòng trong file en và vi không khớp!"
    return en_lines, vi_lines

# Đường dẫn tới file
data_dir = "/content/drive/MyDrive/TiengAnhquaTiengViet/data"
train_en, train_vi = load_parallel_data(os.path.join(data_dir, "train.en"), os.path.join(data_dir, "train.vi"))
test2012_en, test2012_vi = load_parallel_data(os.path.join(data_dir, "tst2012.en"), os.path.join(data_dir, "tst2012.vi"))
test2013_en, test2013_vi = load_parallel_data(os.path.join(data_dir, "tst2013.en"), os.path.join(data_dir, "tst2013.vi"))

# Tạo dataset
train_data = {"translation": [{"en": en, "vi": vi} for en, vi in zip(train_en, train_vi)]}
test2012_data = {"translation": [{"en": en, "vi": vi} for en, vi in zip(test2012_en, test2012_vi)]}
test2013_data = {"translation": [{"en": en, "vi": vi} for en, vi in zip(test2013_en, test2013_vi)]}

train_dataset = Dataset.from_dict(train_data)
eval_dataset = Dataset.from_dict(test2012_data)  # Dùng tst2012 làm tập đánh giá
test_dataset = Dataset.from_dict(test2013_data)  # Dùng tst2013 làm tập kiểm tra cuối

# 2. Tiền xử lý dữ liệu
model_name = "Helsinki-NLP/opus-mt-en-vi"
tokenizer = MarianTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["vi"] for ex in examples["translation"]]

    # Tokenize tiếng Anh (input)
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")

    # Tokenize tiếng Việt (target)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Áp dụng tiền xử lý
train_dataset = train_dataset.map(preprocess_function, batched=True)
eval_dataset = eval_dataset.map(preprocess_function, batched=True)
test_dataset = test_dataset.map(preprocess_function, batched=True)

# 3. Tải mô hình
model = MarianMTModel.from_pretrained(model_name)

# 4. Cấu hình huấn luyện
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=3,
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

# 5. Khởi tạo Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

# 6. Huấn luyện mô hình
trainer.train()

# 7. Lưu mô hình
model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")



/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Map:   0%|          | 0/133317 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3980: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/1553 [00:00<?, ? examples/s]

Map:   0%|          | 0/1268 [00:00<?, ? examples/s]

<ipython-input-5-e1d8fe8823bd>:74: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,0.311200,0.268245
2,0.282900,0.266668
3,0.270900,0.266672


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3339: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[53684]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument index in method wrapper_CUDA__index_select)

#Đánh giá BLEU score


In [ ]:
import torch
from transformers import MarianMTModel, MarianTokenizer
from datasets import Dataset
import sacrebleu
import os

def evaluate_model(model, tokenizer, dataset):
    """
    Đánh giá mô hình trên tập dữ liệu và tính BLEU score.

    Args:
        model: Mô hình MarianMT đã load.
        tokenizer: Tokenizer tương ứng.
        dataset: Dataset chứa các cặp câu tiếng Anh - tiếng Việt.

    Returns:
        float: BLEU score.
    """
    # Xác định thiết bị (GPU nếu có, nếu không thì CPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Chuyển mô hình lên thiết bị
    model.to(device)
    model.eval()

    predictions = []
    references = []

    with torch.no_grad():
        for example in dataset:
            # Chuẩn bị đầu vào và chuyển lên cùng thiết bị
            inputs = tokenizer(example["translation"]["en"], return_tensors="pt", padding=True, truncation=True, max_length=128)
            inputs = {k: v.to(device) for k, v in inputs.items()}  # Chuyển tensor lên GPU/CPU

            # Dự đoán bản dịch
            translated_tokens = model.generate(**inputs, num_beams=5, max_length=128)
            pred = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
            predictions.append(pred)
            references.append([example["translation"]["vi"]])

    # Tính BLEU score
    bleu = sacrebleu.corpus_bleu(predictions, references)
    return bleu.score

# Sử dụng hàm để đánh giá trên tập tst2013
if __name__ == "__main__":
    # Load mô hình và tokenizer
    model_path = "./fine_tuned_model"
    try:
        tokenizer = MarianTokenizer.from_pretrained(model_path)
        model = MarianMTModel.from_pretrained(model_path)
    except Exception as e:
        print(f"Lỗi khi load mô hình: {str(e)}")
        exit()

    # Load dữ liệu tst2013 (giả định đã có từ code trước)
    data_dir = "/content/drive/MyDrive/TiengAnhquaTiengViet/data"
    test2013_en, test2013_vi = [], []
    with open(os.path.join(data_dir, "tst2013.en"), 'r', encoding='utf-8') as f_en:
        test2013_en = [line.strip() for line in f_en]
    with open(os.path.join(data_dir, "tst2013.vi"), 'r', encoding='utf-8') as f_vi:
        test2013_vi = [line.strip() for line in f_vi]

    assert len(test2013_en) == len(test2013_vi), "Số dòng trong tst2013.en và tst2013.vi không khớp!"

    test_data = {"translation": [{"en": en, "vi": vi} for en, vi in zip(test2013_en, test2013_vi)]}
    test_dataset = Dataset.from_dict(test_data)

    # Đánh giá
    try:
        bleu_score = evaluate_model(model, tokenizer, test_dataset)
        print(f"BLEU score trên tập tst2013: {bleu_score:.2f}")
    except Exception as e:
        print(f"Lỗi khi đánh giá: {str(e)}")

BLEU score trên tập tst2013: 48.44


In [13]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.6 MB/s eta 0:00:00


In [16]:
import torch
from transformers import MarianMTModel, MarianTokenizer
import sacrebleu

# 1. Load your fine-tuned model and tokenizer
model_path = "/content/drive/MyDrive/TiengAnhquaTiengViet/model_and_results/fine_tuned_model"  # Replace with your model path
tokenizer = MarianTokenizer.from_pretrained(model_path)
model = MarianMTModel.from_pretrained(model_path)

# 2. Define a function to translate and calculate BLEU
def evaluate_translation(model, tokenizer, source_text, reference_text):
    """Translates source text and calculates BLEU score against reference."""
    inputs = tokenizer(source_text, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        translated_tokens = model.generate(**inputs)

    translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

    bleu = sacrebleu.sentence_bleu(translated_text, [reference_text])
    return bleu.score

# 3. Example usage
source_text = "The cat is sitting on the mat."
reference_text = "Con mèo đang ngồi trên tấm thảm."  # Replace with your actual reference

bleu_score = evaluate_translation(model, tokenizer, source_text, reference_text)
print(f"BLEU Score: {bleu_score:.4f}")

BLEU Score: 61.2975


In [ ]:
!zip -r /content/model_and_results.zip fine_tuned_model results

  adding: fine_tuned_model/ (stored 0%)
  adding: fine_tuned_model/vocab.json (deflated 70%)
  adding: fine_tuned_model/model.safetensors (deflated 7%)
  adding: fine_tuned_model/tokenizer_config.json (deflated 68%)
  adding: fine_tuned_model/generation_config.json (deflated 43%)
  adding: fine_tuned_model/special_tokens_map.json (deflated 35%)
  adding: fine_tuned_model/source.spm (deflated 51%)
  adding: fine_tuned_model/target.spm (deflated 50%)
  adding: fine_tuned_model/config.json (deflated 63%)
  adding: results/ (stored 0%)
  adding: results/runs/ (stored 0%)
  adding: results/runs/Apr18_11-21-05_6d50deae6e61/ (stored 0%)
  adding: results/runs/Apr18_11-21-05_6d50deae6e61/events.out.tfevents.1744975269.6d50deae6e61.1521.0 (deflated 63%)
  adding: results/checkpoint-16666/ (stored 0%)
  adding: results/checkpoint-16666/rng_state.pth (deflated 25%)
  adding: results/checkpoint-16666/vocab.json (deflated 70%)
  adding: results/checkpoint-16666/training_args.bin (deflated 52%)
  ad

#Dịch

In [ ]:
# Install required libraries
!pip install transformers torch ipywidgets

###Dịch không tính Bleu score


In [32]:
from transformers import MarianMTModel, MarianTokenizer
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output

def translate_english_to_vietnamese(sentence, model_path="/content/drive/MyDrive/TiengAnhquaTiengViet/model_and_results/fine_tuned_model"):
    try:
        tokenizer = MarianTokenizer.from_pretrained(model_path)
        model = MarianMTModel.from_pretrained(model_path)
    except Exception as e:
        raise Exception(f"Không thể load mô hình từ {model_path}. Lỗi: {str(e)}")

    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=128)
    model.eval()
    with torch.no_grad():
        translated_tokens = model.generate(**inputs, num_beams=5, max_length=128)
    translated_sentence = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    return translated_sentence

# Create UI elements
english_input = widgets.Textarea(
    value='',
    placeholder='Nhập câu tiếng Anh',
    description='Tiếng Anh:',
    layout={'width': '800px', 'height': '100px'},  # Đặt chiều cao rõ ràng
    rows=3  # Số dòng hiển thị
)

translate_button = widgets.Button(
    description='Dịch',
    button_style='primary',
    tooltip='Nhấn để dịch',
    icon='language'
)

vietnamese_output = widgets.Textarea(
    value='',
    placeholder='Bản dịch tiếng Việt sẽ hiển thị ở đây',
    description='Tiếng Việt:',
    disabled=True,
    layout={'width': '800px', 'height': '100px'},  # Đặt chiều cao rõ ràng
    rows=3  # Số dòng hiển thị
)

output_area = widgets.Output()

# Define button click handler
def on_translate_button_clicked(b):
    with output_area:
        clear_output()
        if not english_input.value.strip():
            print("Vui lòng nhập một câu tiếng Anh hợp lệ.")
            vietnamese_output.value = ""
            return
        try:
            result = translate_english_to_vietnamese(english_input.value.strip())
            vietnamese_output.value = result
        except Exception as e:
            print(f"Lỗi: {str(e)}")
            vietnamese_output.value = ""

# Bind the handler to the button
translate_button.on_click(on_translate_button_clicked)

# Display the UI
display(english_input)
display(translate_button)
display(vietnamese_output)
display(output_area)

Textarea(value='', description='Tiếng Anh:', layout=Layout(height='100px', width='800px'), placeholder='Nhập c…

Button(button_style='primary', description='Dịch', icon='language', style=ButtonStyle(), tooltip='Nhấn để dịch…

Textarea(value='', description='Tiếng Việt:', disabled=True, layout=Layout(height='100px', width='800px'), pla…

Output()

###Dịch có tính Bleu Score

In [33]:
from transformers import MarianMTModel, MarianTokenizer
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output
import sacrebleu

def translate_english_to_vietnamese(sentence, model_path="/content/drive/MyDrive/TiengAnhquaTiengViet/model_and_results/fine_tuned_model"):
    try:
        tokenizer = MarianTokenizer.from_pretrained(model_path)
        model = MarianMTModel.from_pretrained(model_path)
    except Exception as e:
        raise Exception(f"Không thể load mô hình từ {model_path}. Lỗi: {str(e)}")

    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=128)
    model.eval()
    with torch.no_grad():
        translated_tokens = model.generate(**inputs, num_beams=5, max_length=128)
    translated_sentence = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    return translated_sentence, translated_tokens[0]

# UI elements
english_input = widgets.Textarea(
    value='',
    placeholder='Nhập câu tiếng Anh',
    description='Tiếng Anh:',
    layout={'width': '800px', 'height': '100px'},
    rows=3
)

reference_input = widgets.Textarea(  # Reference input area
    value='',
    placeholder='Nhập bản dịch tham chiếu tiếng Việt',
    description='Tham chiếu:',
    layout={'width': '800px', 'height': '100px'},
    rows=3
)

translate_button = widgets.Button(
    description='Dịch',
    button_style='primary',
    tooltip='Nhấn để dịch',
    icon='language'
)

vietnamese_output = widgets.Textarea(
    value='',
    placeholder='Bản dịch tiếng Việt sẽ hiển thị ở đây',
    description='Tiếng Việt:',
    disabled=True,
    layout={'width': '800px', 'height': '100px'},
    rows=3
)

output_area = widgets.Output()

# Button click handler
def on_translate_button_clicked(b):
    with output_area:
        clear_output()
        if not english_input.value.strip():
            print("Vui lòng nhập một câu tiếng Anh hợp lệ.")
            vietnamese_output.value = ""
            return
        if not reference_input.value.strip():  # Check for reference translation
            print("Vui lòng nhập bản dịch tham chiếu tiếng Việt.")
            vietnamese_output.value = ""
            return

        try:
            result, translated_tokens = translate_english_to_vietnamese(english_input.value.strip())
            vietnamese_output.value = result

            # Calculate and display BLEU score
            reference_translation = reference_input.value.strip()  # Get reference from input area
            bleu = sacrebleu.sentence_bleu(
                vietnamese_output.value,
                [reference_translation],
            )

            print(f"BLEU Score: {bleu.score:.4f}")

        except Exception as e:
            print(f"Lỗi: {str(e)}")
            vietnamese_output.value = ""

# Bind handler and display UI
translate_button.on_click(on_translate_button_clicked)
display(english_input)
display(reference_input)  # Display reference input area
display(translate_button)
display(vietnamese_output)
display(output_area)

Textarea(value='', description='Tiếng Anh:', layout=Layout(height='100px', width='800px'), placeholder='Nhập c…

Textarea(value='', description='Tham chiếu:', layout=Layout(height='100px', width='800px'), placeholder='Nhập …

Button(button_style='primary', description='Dịch', icon='language', style=ButtonStyle(), tooltip='Nhấn để dịch…

Textarea(value='', description='Tiếng Việt:', disabled=True, layout=Layout(height='100px', width='800px'), pla…

Output()

##Chạy đánh giá trên tập test khác

In [ ]:
import torch
from transformers import MarianMTModel, MarianTokenizer
from datasets import Dataset
import sacrebleu
import os

def evaluate_on_test_set(test_en_file, test_vi_file, model_path="./fine_tuned_model"):
    """
    Đánh giá mô hình trên tập test mới và tính BLEU score.

    Args:
        test_en_file (str): Đường dẫn tới file chứa các câu tiếng Anh.
        test_vi_file (str): Đường dẫn tới file chứa các câu tiếng Việt (tham chiếu).
        model_path (str): Đường dẫn tới thư mục chứa mô hình và tokenizer.

    Returns:
        float: BLEU score trên tập test.
    """
    # 1. Load mô hình và tokenizer
    try:
        tokenizer = MarianTokenizer.from_pretrained(model_path)
        model = MarianMTModel.from_pretrained(model_path)
    except Exception as e:
        raise Exception(f"Không thể load mô hình từ {model_path}. Lỗi: {str(e)}")

    # 2. Đọc dữ liệu test
    def load_parallel_data(en_file, vi_file):
        with open(en_file, 'r', encoding='utf-8') as f_en, open(vi_file, 'r', encoding='utf-8') as f_vi:
            en_lines = [line.strip() for line in f_en]
            vi_lines = [line.strip() for line in f_vi]
        assert len(en_lines) == len(vi_lines), "Số dòng trong file en và vi không khớp!"
        return en_lines, vi_lines

    test_en, test_vi = load_parallel_data(test_en_file, test_vi_file)

    # 3. Tạo dataset
    test_data = {"translation": [{"en": en, "vi": vi} for en, vi in zip(test_en, test_vi)]}
    test_dataset = Dataset.from_dict(test_data)

    # 4. Tiền xử lý dữ liệu
    def preprocess_function(examples):
        inputs = [ex["en"] for ex in examples["translation"]]
        model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length", return_tensors="pt")
        return model_inputs

    test_dataset = test_dataset.map(preprocess_function, batched=True)

    # 5. Dự đoán và tính BLEU score
    model.eval()
    predictions = []
    references = []

    with torch.no_grad():
        for example in test_dataset:
            inputs = {
                "input_ids": torch.tensor([example["input_ids"]]),
                "attention_mask": torch.tensor([example["attention_mask"]])
            }
            translated_tokens = model.generate(**inputs, num_beams=5, max_length=128)
            pred = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
            predictions.append(pred)
            references.append([example["translation"]["vi"]])

    # Tính BLEU score
    bleu = sacrebleu.corpus_bleu(predictions, references)
    return bleu.score

# Sử dụng hàm
if __name__ == "__main__":
    # Đường dẫn tới tập test mới
    test_en_file = "./data/test.en"  # Thay bằng đường dẫn thực tế
    test_vi_file = "./data/test.vi"  # Thay bằng đường dẫn thực tế

    try:
        bleu_score = evaluate_on_test_set(test_en_file, test_vi_file)
        print(f"BLEU score trên tập test mới: {bleu_score:.2f}")
    except Exception as e:
        print(f"Lỗi: {str(e)}")